# Mapping the Margins: R2 reproducibility rerun (Kaggle, free tier)

Regenerates the **E4 subset** (class x race, 432 prompts) through **both** confirmatory models on
Kaggle's free-tier T4s, as pre-registered on 16 Oct 2026 (`docs/preregistration.md` §10). It runs
the repository's own pipeline at the pinned tag `r2-v1`; nothing is measured here.

**Before running**
1. *Settings -> Accelerator -> GPU T4 x2*, and *Internet -> On* (needs a phone-verified account).
2. *Add-ons -> Secrets*: attach **`HF_TOKEN`** (read; the account must have accepted the Llama 3.1
   licence). The repository is public, so no GitHub token is needed; an optional `GITHUB_TOKEN`
   is used only if attached, and never appears in a command or an error message.
3. Run with **Save Version -> Save & Run All (Commit)**, so it keeps running with the tab closed.
   Expect ~1 h of downloads and setup, then roughly 3-5 h of generation (both models at once).

**Output**: `r2_kaggle_outputs.tar.gz` in the notebook's *Output* tab, plus one log per model.
Download it and hand it back; `run_p11.py r2-process --machine kaggle` analyses it on any CPU.

If a step fails, the notebook stops with the reason. Do not edit the cells to work around it:
never unpin, never `pip install`, never swap a model. Report the last ~50 lines of the cell.


In [ ]:
import os, subprocess, time
from pathlib import Path

REPO = 'github.com/breenu/mapping-the-margins.git'
REF = 'r2-v1'
MODELS = ['llama31_8b_instruct', 'mistral7b_instruct_v03']
WORK = Path("/tmp/mapping-the-margins")
OUT = Path("/kaggle/working")


def sh(command, cwd=None):
    print(f"\n$ {command}", flush=True)
    subprocess.run(command, shell=True, check=True, cwd=cwd)


try:
    from kaggle_secrets import UserSecretsClient

    _secrets = UserSecretsClient()
    # .strip(): a secret pasted with a leading space broke the clone URL on the first real run.
    os.environ["HF_TOKEN"] = _secrets.get_secret("HF_TOKEN").strip()
except Exception as error:
    message = "Attach the HF_TOKEN secret (Add-ons -> Secrets)."
    raise SystemExit(message) from error
try:
    # Optional since the repository went public (17 Oct 2026); needed only for a private fork.
    _github_token = _secrets.get_secret("GITHUB_TOKEN").strip()
except Exception:
    _github_token = ""

if not WORK.exists():
    # The token NEVER appears in a command line: a failed subprocess prints its argv, and on the
    # first real run that printed the token. It travels as an HTTP header set through git's
    # GIT_CONFIG_* environment variables, scoped to this one clone, and any error text is
    # scrubbed of it before it is shown.
    import base64

    _env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    _basic = ""
    if _github_token:
        _basic = base64.b64encode(f"x-access-token:{_github_token}".encode()).decode()
        _env.update(
            {
                "GIT_CONFIG_COUNT": "1",
                "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
                "GIT_CONFIG_VALUE_0": f"Authorization: Basic {_basic}",
            }
        )
    _clone = subprocess.run(
        ["git", "clone", "--quiet", f"https://{REPO}", str(WORK)],
        env=_env,
        capture_output=True,
        text=True,
    )
    if _clone.returncode != 0:
        _detail = _clone.stderr or ""
        for _secret in (_github_token, _basic):
            if _secret:  # str.replace("", ...) would insert *** between every character
                _detail = _detail.replace(_secret, "***")
        raise SystemExit(
            f"git clone failed (exit {_clone.returncode}). If a GITHUB_TOKEN is attached, it "
            f"must have Contents: Read-only on the repository.\n{_detail[-1500:]}"
        )
sh(f"git checkout --quiet {REF}", cwd=WORK)
sh("git log -1 --format='%H %s'", cwd=WORK)


In [ ]:
# The LOCKED environment, exactly as the lab installed it. `--locked` fails rather than resolve.
sh("curl -LsSf https://astral.sh/uv/install.sh | sh")
os.environ["PATH"] = f"{Path.home()}/.local/bin:" + os.environ["PATH"]
sh("uv sync --locked --extra inference", cwd=WORK)
sh("nvidia-smi")


In [ ]:
# The prompt files, built ONCE and BEFORE the GPU check: `verify-gpu` times a batch of E1 prompts
# (a fresh clone has none -- the first Kaggle run stopped exactly there), and the two parallel R2
# runs below must not both rewrite R2's files.
for command in ['uv run python scripts/build_prompts.py --prompt-set open_gen_v1 --decoding sampled_v1 --out-dir data/prompts/e1_sampled --run-name build-prompts-e1-sampled', 'uv run python scripts/build_prompts.py --prompt-set open_gen_v1 --decoding sampled_v1 --pair socioeconomic_class race_ethnicity --out-dir data/prompts/r2_sampled --run-name build-prompts-r2-sampled', 'uv run python scripts/build_prompts.py --prompt-set open_gen_v1 --decoding greedy_v1 --pair socioeconomic_class race_ethnicity --out-dir data/prompts/e4_greedy --run-name build-prompts-e4-greedy']:
    sh(command, cwd=WORK)


In [ ]:
# Both models must load at NF4 on a T4 and generate, or nothing later is the measurement.
# (First run downloads ~30 GB of weights.)
for model in MODELS:
    sh(
        f"CUDA_VISIBLE_DEVICES=0 uv run python scripts/run_p11.py verify-gpu --model {model}",
        cwd=WORK,
    )


In [ ]:
# One model per T4, in parallel. Each run is the lab's own `r2` step (resumable, preflighted).
running = {}
for gpu, model in enumerate(MODELS):
    log = open(OUT / f"r2_{model}.log", "w")
    command = (
        f"CUDA_VISIBLE_DEVICES={gpu} uv run python scripts/run_p11.py r2 --model {model} "
        f"--machine kaggle --no-prompt-build"
    )
    print(f"$ {command}  > {log.name}", flush=True)
    process = subprocess.Popen(
        command, shell=True, cwd=WORK, stdout=log, stderr=subprocess.STDOUT
    )
    running[model] = (process, log)

responses = WORK / "data" / "responses"
while any(process.poll() is None for process, _ in running.values()):
    time.sleep(300)
    counts = {
        f"{path.parent.parent.name}/{path.parent.name}": len(
            path.read_text(encoding="utf-8").splitlines()
        )
        for path in sorted(responses.glob("r2_kaggle_*/*/*.jsonl"))
    }
    print(time.strftime("%H:%M"), counts, flush=True)

for model, (process, log) in running.items():
    log.close()
    tail = Path(log.name).read_text(encoding="utf-8").splitlines()[-25:]
    print(f"\n==== {model}: exit {process.returncode} ====", *tail, sep="\n")
failed = [model for model, (process, _) in running.items() if process.returncode]
if failed:
    raise SystemExit(f"R2 generation FAILED for {failed}; see the logs above. Report them.")


In [ ]:
sh(
    "tar czf /kaggle/working/r2_kaggle_outputs.tar.gz "
    "data/responses/r2_kaggle_sampled data/responses/r2_kaggle_greedy data/p11 results/run_manifests",
    cwd=WORK,
)
sh("ls -lh /kaggle/working")
